<a href="https://colab.research.google.com/github/john-dechellis-weather/wx_compare/blob/main/wx_compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y libeccodes0 > /dev/null 2>&1
!pip install -q cfgrib xarray eccodes requests pandas matplotlib

In [ ]:
import os, sys

REPO_URL = "https://github.com/john-dechellis-weather/wx_compare.git"
REPO_DIR = "/content/wx_compare"

os.chdir('/content')
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

In [3]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
CACHE_ROOT = Path('/content/drive/MyDrive/wx_compare_cache')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
raw = input("Enter ICAO codes (comma-separated, e.g. KJFK,KORD,KSEA): ")
ICAOS = [s.strip().upper() for s in raw.split(",") if s.strip()]

cycle_choice = input(
    "Cycle? Enter 'auto' for latest complete, or HH (00/06/12/18) for today's cycle: "
).strip().lower()

print(f"Stations: {ICAOS}")
print(f"Cycle selection: {cycle_choice}")

In [ ]:
from datetime import datetime, timezone, timedelta
from compare import compare_icaos
from core.cycle_select import find_latest_complete
from core.stations import StationResolver
from models import GfsMos, Hrrr

if cycle_choice == "auto":
    resolver = StationResolver(cache_dir=CACHE_ROOT / "stations")
    resolved_pre, _ = resolver.resolve_many(ICAOS)
    probe_sources = [
        GfsMos(cache_dir=CACHE_ROOT / "gfs_mos"),
        Hrrr(cache_dir=CACHE_ROOT / "hrrr", stations=resolved_pre, fhours=range(0, 19)),
    ]
    print("Probing NOMADS for latest complete cycle...")
    CYCLE = find_latest_complete(probe_sources)
    if CYCLE is None:
        raise RuntimeError("No complete cycle found")
else:
    today = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    CYCLE = today.replace(hour=int(cycle_choice))
    if CYCLE > datetime.now(timezone.utc):
        CYCLE = CYCLE - timedelta(days=1)
    print(f"Using cycle: {CYCLE:%Y-%m-%d %HZ}")

df, resolved, unresolved = compare_icaos(
    icaos=ICAOS, cycle=CYCLE, cache_root=CACHE_ROOT,
)

if unresolved:
    print(f"⚠ Not found: {unresolved}")

print(f"\nResolved {len(resolved)} stations:")
for s in resolved:
    print(f"  {s.icao}  {s.name}  ({s.lat:.2f}, {s.lon:.2f}, {s.elev_ft:.0f} ft)")

df

In [ ]:
# Basic Plot #

from compare import plot_comparison
for s in resolved:
    plot_comparison(df, s.icao)

In [ ]:
# Interactive Plot #

from compare import plot_comparison_interactive
for s in resolved:
    plot_comparison_interactive(df, s.icao).show()